# 08. Гибрид: формула и двухэтапная схема (E3)

Контекст — `решения.md`: 3.3 (гибрид), 3.7 (E3), итоги E1–E2 (3.2, 3.7).

- **Данные — как в E1:** подвыборка 20% (seed 42), обучение до 2021, валидация 2021, тест — с 2022, тест — один раз.
- **Два каталога для content-части** (06, раздел 5): **«известные на дату»** (первая оценка в полных данных до даты
  разбиения) — основной режим; **«весь каталог»** — как в E1, верхняя граница (там есть книги, впервые оценённые после даты).
  Во втором режиме ранжировщик — без года издания: по нему он узнаёт «будущие» книги.
- **Пользователи валидации делятся пополам (seed 42):** на половине A подбирается всё (N профиля content, веса формулы,
  ранжировщик), на половине B — честное сравнение на валидации. Для теста веса формулы подбираются на всей валидации,
  ранжировщик учится на всей валидации с числом деревьев из A.
- **Без утечки в тест:** ранжировщик учится на признаках по обучению (до 2021) и ответах 2021 года; на тесте признаки —
  по обучению + валидации (до 2022), каталог — известные до 2022, оценки 2022–2023 не используются нигде, кроме проверки.
- **Источники кандидатов:** ItemKNN (BM25, E1), content TF-IDF и e5-base с профилем по последним N оценкам (E2), по 100
  книг; ALS — признак (балл для любой пары), в кандидаты — если заметно добавляет попаданий.
- **Без книг с 1–4 оценками в обучении** — ни в кандидатах, ни в одиночных моделях (`candidates.drop_survivors`):
  подвыборка оставляет книги с ≥ 5 оценками за всё время, и такая книга попала в данные только благодаря будущим оценкам
  (попадание в 2021 — у 89–95% книг с 1–2 оценками против 11% у книг с 5–9). Ранжировщик выучил бы это по числу оценок.
  ItemKNN из-за этого чуть ниже, чем в E1 (0,0105 против 0,0111 на валидации), у ALS таких книг в выдаче нет.
- Код — `ml/src/hybrid/` (кандидаты и признаки, формула, ранжировщик).

In [1]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import gc
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
from rectools import Columns

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent / "src"))
from cf.models import SEED, fit_model
from cf.tune import STORAGE, best_params
from content.embeddings import catalog, load_books
from content.recommend import recommend, tfidf_vectors
from eval.coldstart import truncate_history
from eval.data import TEST_START, VAL_START, first_seen, known_at, load_ratings, subsample_users, time_split
from eval.metrics import K, bootstrap_ci, eval_users, evaluate, paired_bootstrap, with_ci
from eval.search import informative
from hybrid import blend, candidates, ranker

U, I = Columns.User, Columns.Item
REPORTS = Path.cwd().parent / "reports" / "metrics"
pd.options.display.float_format = "{:.4f}".format
K_LIST = 100

ratings = load_ratings()
first = first_seen(ratings)
sub = subsample_users(ratings, 0.2, seed=SEED)
del ratings
train, val, test = time_split(sub)
train_full = pd.concat([train, val])
users_val = eval_users(train, val)
users_test = eval_users(train_full, test)
perm = np.random.default_rng(SEED).permutation(users_val)
A, B = np.sort(perm[: len(perm) // 2]), np.sort(perm[len(perm) // 2:])

books = load_books(["parent_asin", "author", "year", "cat2", "cat3", "text", "features", "categories"])
ids_all = books["parent_asin"].to_numpy()
allowed_all = informative(books)
X_tfidf, _ = tfidf_vectors(books["text"])
E5 = np.asarray(catalog("e5-base"))
ctx = {"X_tfidf": X_tfidf, "E5": E5, "ids_all": ids_all}
VECTORS = {"tfidf": X_tfidf, "e5": E5}
REGIMES = ["известные", "весь"]

def catalog_mask(regime, cutoff):
    return allowed_all & known_at(ids_all, first, cutoff) if regime == "известные" else allowed_all

optuna.logging.set_verbosity(optuna.logging.WARNING)
BEST = {k: best_params(optuna.load_study(study_name=k, storage=f"sqlite:///{STORAGE}")) for k in ["knn", "als"]}
print(f"Валидация: {len(users_val):,} пользователей (A {len(A):,}, B {len(B):,}); тест: {len(users_test):,}")

Валидация: 21,350 пользователей (A 10,675, B 10,675); тест: 20,009


## 1. Списки моделей на валидации и N профиля content

Каждая модель выдаёт по 100 книг всем пользователям валидации. N профиля content (последние N оценок) выбирается по
NDCG@10 на половине A, отдельно для TF-IDF и e5-base и для каждого каталога (первые 10 из 100 — это и есть топ-10 модели).

In [2]:
timing = []
def timed(stage, part, f):
    t0 = time.time()
    out = f()
    timing.append({"этап": stage, "часть": part, "с": round(time.time() - t0, 1)})
    return out

def cf_lists(tr, users, part):
    knn, ds_knn = timed("обучение ItemKNN", part, lambda: fit_model("knn", BEST["knn"], tr))
    als = timed("обучение ALS", part, lambda: fit_model("als", BEST["als"], tr))
    out = {"knn": timed("кандидаты ItemKNN", part, lambda: knn.recommend(users, ds_knn, k=K_LIST, filter_viewed=True,
                                                                       on_unsupported_targets="ignore")),
           "als": timed("кандидаты ALS", part, lambda: als[0].recommend(users, als[1], k=K_LIST, filter_viewed=True,
                                                                      on_unsupported_targets="ignore"))}
    out = {s: candidates.drop_survivors(lst, tr) for s, lst in out.items()}
    return out, als

def head(lst, users, k=K):
    return lst[(lst[Columns.Rank] <= k) & lst[U].astype(str).isin(set(users))]

lists_val, als_val = cf_lists(train, users_val, "val")
N_GRID = {"tfidf": [1, 3, 5], "e5": [1, 2, 3, 5]}
content_val, prof_rows = {}, []
for regime in REGIMES:
    mask = catalog_mask(regime, VAL_START)
    for s, Ns in N_GRID.items():
        for n in Ns:
            lst = timed(f"кандидаты {s}", "val", lambda: recommend(VECTORS[s], ids_all, train, users_val, k=K_LIST, allowed=mask,
                                                                profile_from=truncate_history(train, users_val, n)))
            content_val[(regime, s, n)] = lst = candidates.drop_survivors(lst, train)
            t, _ = evaluate(head(lst, A), train, val, A)
            prof_rows.append({"каталог": regime, "модель": s, "N": n, **dict(zip(t["разрез"], t["NDCG@10"]))})
prof = pd.DataFrame(prof_rows)
N_BEST = {(r, s): int(prof[(prof["каталог"] == r) & (prof["модель"] == s)].sort_values("все").iloc[-1]["N"])
          for r in REGIMES for s in N_GRID}
for key in [k for k in content_val if k[2] != N_BEST[(k[0], k[1])]]:   # невыбранные N больше не нужны
    del content_val[key]
display(prof.pivot_table(index=["каталог", "модель"], columns="N", values="все", sort=False))
N_BEST

N                     1      3      5      2
каталог   модель                            
известные tfidf  0.0055 0.0059 0.0059    NaN
          e5     0.0059 0.0055 0.0049 0.0056
весь      tfidf  0.0143 0.0156 0.0147    NaN
          e5     0.0156 0.0145 0.0126 0.0153

{('известные', 'tfidf'): 3,
 ('известные', 'e5'): 1,
 ('весь', 'tfidf'): 3,
 ('весь', 'e5'): 1}

**Профиль content (подбор на половине A):** лучший N — **3 для TF-IDF и 1 для e5-base** в обоих режимах каталога
(«известные»: TF-IDF 0,0055 / 0,0059 / 0,0059 при N = 1 / 3 / 5, e5-base 0,0059 при N = 1 против 0,0049 при N = 5;
весь каталог — 0,0156 и 0,0156). Разница между соседними N мала, но вся история заметно хуже (E2), — короткий профиль
по последним оценкам и идёт в кандидаты и признаки.

## 2. Кандидаты и признаки (валидация)

In [3]:
def candidate_recall(lists, part, users, sources, k=K_LIST):
    hits = part[part["rating"] >= 4]
    hits = set(zip(hits[U].astype(str), hits[I].astype(str)))
    pool = pd.concat([lists[s][lists[s][Columns.Rank] <= k][[U, I]].astype(str) for s in sources]).drop_duplicates()
    pool = pool[pool[U].isin(set(users))]
    found = sum((u, i) in hits for u, i in zip(pool[U], pool[I]))
    return found / candidates.n_relevant(part, users).sum(), len(pool) / len(users)

def lists_for(regime, lists_cf, content):
    return {**lists_cf, "tfidf": content[(regime, "tfidf", N_BEST[(regime, "tfidf")])],
            "e5": content[(regime, "e5", N_BEST[(regime, "e5")])]}

rec_rows = []
for regime in REGIMES:
    L = lists_for(regime, lists_val, content_val)
    for sources in [["knn"], ["tfidf"], ["e5"], ["knn", "tfidf"], ["knn", "tfidf", "e5"], ["knn", "tfidf", "e5", "als"]]:
        for k in [50, 100]:
            r, size = candidate_recall(L, val, A, sources, k)
            rec_rows.append({"каталог": regime, "источники": " + ".join(sources), "k": k, "доля попаданий в кандидатах": r,
                             "кандидатов на пользователя": size})
rec = pd.DataFrame(rec_rows)
display(rec.pivot_table(index=["каталог", "источники"], columns="k", values=["доля попаданий в кандидатах", "кандидатов на пользователя"], sort=False))

# ALS в кандидаты — если добавляет ≥ 5% попаданий (относительно) в основном режиме
r3 = rec.query("каталог == 'известные' and k == 100 and источники == 'knn + tfidf + e5'")["доля попаданий в кандидатах"].iloc[0]
r4 = rec.query("каталог == 'известные' and k == 100 and источники == 'knn + tfidf + e5 + als'")["доля попаданий в кандидатах"].iloc[0]
SOURCES = ["knn", "tfidf", "e5"] + (["als"] if r4 / r3 - 1 >= 0.05 else [])
print(f"ALS добавляет {r4 / r3 - 1:+.1%} попаданий → источники кандидатов: {SOURCES}")

доля попаданий в кандидатах         \
k                                                        50     100   
каталог   источники                                                   
известные knn                                         0.0467 0.0646   
          tfidf                                       0.0193 0.0240   
          e5                                          0.0141 0.0169   
          knn + tfidf                                 0.0551 0.0740   
          knn + tfidf + e5                            0.0583 0.0777   
          knn + tfidf + e5 + als                      0.0748 0.1039   
весь      knn                                         0.0467 0.0646   
          tfidf                                       0.0468 0.0559   
          e5                                          0.0328 0.0368   
          knn + tfidf                                 0.0830 0.1066   
          knn + tfidf + e5                            0.0919 0.1165   
          knn + tfidf + e5 + als                      0.1086 0.1428   

                                 кандидатов на пользователя           
k                                                       50       100  
каталог   источники                                                   
известные knn                                       49.6582  96.9502  
          tfidf                                     50.0000  97.7638  
          e5                                        50.0000  97.4674  
          knn + tfidf                               96.4674 189.7084  
          knn + tfidf + e5                         138.9241 274.2428  
          knn + tfidf + e5 + als                   172.4644 340.3061  
весь      knn                                       49.6582  96.9502  
          tfidf                                     50.0000  98.1470  
          e5                                        50.0000  97.9078  
          knn + tfidf                               96.6393 190.3975  
          knn + tfidf + e5                         138.8420 275.0804  
          knn + tfidf + e5 + als                   172.4546 341.3079

ALS добавляет +33.7% попаданий → источники кандидатов: ['knn', 'tfidf', 'e5', 'als']


In [4]:
def build_cands(regime, lists, als, tr, part, users, cutoff):
    L = {s: lists[s][lists[s][U].astype(str).isin(set(users))] for s in lists}
    c = candidates.build(L, tr, users, books, ctx, cutoff,
                         {s: N_BEST[(regime, s)] for s in ["tfidf", "e5"]}, als=als, sources=SOURCES)
    return candidates.add_labels(c, part)

cands_val = {}
for regime in REGIMES:
    cands_val[regime] = timed(f"признаки ({regime})", "val",
                              lambda: build_cands(regime, lists_for(regime, lists_val, content_val), als_val, train, val, users_val, VAL_START))
    c = cands_val[regime]
    print(f"{regime}: {len(c):,} пар, {c[U].nunique():,} пользователей, попаданий {int(c['hit'].sum()):,}, признаков {len(ranker.feature_columns(c))}")
cands_val["известные"].head()

известные: 7,270,284 пар, 21,350 пользователей, попаданий 5,250, признаков 26


весь: 7,291,084 пар, 21,350 пользователей, попаданий 7,359, признаков 26


,user_id,item_id,knn_score,knn_rank,als_score,als_rank,tfidf_score,tfidf_rank,e5_score,e5_rank,...,i_age,u_n,u_liked,u_days_last,author_cnt,author_last5,section_share,subsection_share,section,hit
0,AE223J6LBWPONMIKQGHAI3X5GNOQ,B07SB1P1LY,2174.9104,1.0000,0.5116,6.0000,NaN,NaN,NaN,NaN,...,2.0000,16.0000,1.0000,23.0000,0.0000,0.0000,0.0000,0.0000,Reference,0
1,AE223J6LBWPONMIKQGHAI3X5GNOQ,B07SBYXH9R,1958.7195,2.0000,0.5856,2.0000,NaN,NaN,NaN,NaN,...,2.0000,16.0000,1.0000,23.0000,0.0000,0.0000,0.5625,0.5625,"Mystery, Thriller & Suspense",0
2,AE223J6LBWPONMIKQGHAI3X5GNOQ,B08NGPDC15,1506.7743,3.0000,0.4631,12.0000,NaN,NaN,NaN,NaN,...,1.0000,16.0000,1.0000,23.0000,1.0000,0.0000,0.5625,0.5625,"Mystery, Thriller & Suspense",0
3,AE223J6LBWPONMIKQGHAI3X5GNOQ,B07QGLVMXS,1336.8531,4.0000,0.5699,3.0000,NaN,NaN,NaN,NaN,...,2.0000,16.0000,1.0000,23.0000,0.0000,0.0000,0.3750,0.3750,Literature & Fiction,0
4,AE223J6LBWPONMIKQGHAI3X5GNOQ,B079J4G42H,1274.9360,5.0000,0.5205,5.0000,NaN,NaN,NaN,NaN,...,8.0000,16.0000,1.0000,23.0000,0.0000,0.0000,0.5625,0.5625,"Mystery, Thriller & Suspense",0


**Кандидаты (по 100 от каждого источника, доля попаданий валидации, попавших в кандидаты):**
- «известные»: ItemKNN 6,5%; + TF-IDF 7,4%; + e5-base 7,8%; **+ ALS 10,4% (+34%)** — ALS добавлен (порог был +5%).
  На весь каталог — 14,3% (content выбирает книги, впервые оценённые позже, — завышено).
- В среднем ~340 кандидатов на пользователя, 7,3 млн пар на валидации. Потолок для переранжирования — 10% попаданий:
  остальные 90% не попадают ни в один список.

## 3. Способ 1 — формула

Итог = w_knn·c_knn + w_tfidf·c_tfidf + w_e5·c_e5 + w_wr·c_wr, веса ≥ 0, сумма 1 (сетка с шагом 0,1 — 286 наборов), свои в
каждой корзине числа оценок (≤ 5 / 6–19 / 20+). Нормировка внутри пользователя — **ранги** (1 − (место − 1)/100, сумма —
счёт Борда) или **min-max** баллов; порог взвешенного рейтинга m — 5 / 20 / 100. Всё выбирается на A, проверка — на B.

In [5]:
FORMULA_SOURCES = ["knn", "tfidf", "e5"]
WEIGHTS = blend.simplex(len(FORMULA_SOURCES) + 1, 0.1)
COLS = [f"c_{s}" for s in FORMULA_SOURCES] + ["c_wr"]

def with_components(c, method, m, tr, cutoff):
    stats, mean = candidates.item_stats(tr, cutoff, m)
    c = c.drop(columns="i_wr").join(stats["i_wr"], on=I)
    c["i_wr"] = c["i_wr"].fillna(mean)
    return pd.concat([c, blend.components(c, FORMULA_SOURCES, method, K_LIST)], axis=1)

def bucket_users(c):
    u_n = c.groupby(U)["u_n"].first()
    return blend.bucket_of(u_n)

def tune_formula(c, part, users):
    """Лучшие веса для каждой корзины на пользователях users: {корзина: (веса, NDCG)}."""
    c = c[c[U].isin(set(users))]
    buckets = bucket_users(c)
    out = {}
    for _, _, b in blend.BUCKETS:
        cb = c[c[U].isin(set(buckets[buckets == b].index))]
        uu, F, hit, valid = blend.to_matrix(cb, COLS)
        res = blend.grid(F, hit, valid, candidates.n_relevant(part, uu).to_numpy(), WEIGHTS)
        out[b] = (WEIGHTS[res.argmax()], float(res.max()))
    return out

def formula_score(c, weights):
    buckets = bucket_users(c).reindex(c[U]).to_numpy()
    W = np.stack([weights[b][0] for b in buckets])
    return (c[COLS].to_numpy() * W).sum(1)

formula_rows, FORMULA = [], {}
for regime in REGIMES:
    for method in ["rank", "minmax"]:
        for m in [5, 20, 100]:
            cm = with_components(cands_val[regime], method, m, train, VAL_START)
            w = tune_formula(cm, val, A)
            cA = cm[cm[U].isin(set(A))]
            ndcg_A = evaluate(blend.top_k(cA, formula_score(cA, w)), train, val, A)[0]["NDCG@10"].iloc[0]
            formula_rows.append({"каталог": regime, "нормировка": method, "m": m, "NDCG@10 на A": ndcg_A,
                                 **{f"веса {b} (knn/tfidf/e5/wr)": " / ".join(f"{x:.1f}" for x in w[b][0]) for b in w}})
            FORMULA[(regime, method, m)] = w
formula_tab = pd.DataFrame(formula_rows)
display(formula_tab)
BEST_FORMULA = {r: tuple(formula_tab[formula_tab["каталог"] == r].sort_values("NDCG@10 на A").iloc[-1][["нормировка", "m"]])
                for r in REGIMES}
BEST_FORMULA

,каталог,нормировка,m,NDCG@10 на A,веса ≤ 5 (knn/tfidf/e5/wr),веса 6–19 (knn/tfidf/e5/wr),веса 20+ (knn/tfidf/e5/wr)
0,известные,rank,5,0.0129,0.6 / 0.2 / 0.2 / 0.0,0.5 / 0.2 / 0.3 / 0.0,0.5 / 0.4 / 0.0 / 0.1
1,известные,rank,20,0.0129,0.6 / 0.2 / 0.2 / 0.0,0.5 / 0.2 / 0.3 / 0.0,0.5 / 0.4 / 0.0 / 0.1
2,известные,rank,100,0.0129,0.6 / 0.2 / 0.2 / 0.0,0.5 / 0.2 / 0.3 / 0.0,0.5 / 0.4 / 0.0 / 0.1
3,известные,minmax,5,0.0123,0.7 / 0.1 / 0.2 / 0.0,0.4 / 0.1 / 0.2 / 0.3,0.4 / 0.2 / 0.0 / 0.4
4,известные,minmax,20,0.0124,0.3 / 0.1 / 0.1 / 0.5,0.4 / 0.1 / 0.2 / 0.3,0.4 / 0.2 / 0.1 / 0.3
5,известные,minmax,100,0.0124,0.3 / 0.1 / 0.2 / 0.4,0.4 / 0.1 / 0.2 / 0.3,0.4 / 0.2 / 0.2 / 0.2
6,весь,rank,5,0.0183,0.2 / 0.4 / 0.4 / 0.0,0.0 / 0.8 / 0.2 / 0.0,0.0 / 0.7 / 0.3 / 0.0
7,весь,rank,20,0.0183,0.2 / 0.4 / 0.4 / 0.0,0.0 / 0.8 / 0.2 / 0.0,0.0 / 0.5 / 0.4 / 0.1
8,весь,rank,100,0.0183,0.2 / 0.4 / 0.4 / 0.0,0.0 / 0.8 / 0.2 / 0.0,0.0 / 0.5 / 0.4 / 0.1
9,весь,minmax,5,0.0184,0.3 / 0.4 / 0.2 / 0.1,0.0 / 0.6 / 0.4 / 0.0,0.4 / 0.4 / 0.2 / 0.0


{'известные': ('rank', 5), 'весь': ('minmax', 100)}

**Формула w₁·ItemKNN + w₂·TF-IDF + w₃·e5-base + w₄·WR (подбор на A, веса — по корзинам числа оценок):**
- **Нормировка — по местам** (1 − (место − 1)/100, у кого книги нет в списке — 0): в «известных» 0,0129 против
  0,0123–0,0124 у min-max. Баллы моделей в разных шкалах (BM25 — тысячи, косинус — до 1) и с длинным хвостом сверху:
  min-max отдаёт всё одной-двум книгам с выбросом, места устойчивы. На всём каталоге ничья (0,0183 против 0,0185).
- **WR почти не нужен:** вес 0 в двух корзинах из трёх, m (5 / 20 / 100) на результат не влияет.
- **Веса (весь val, «известные»):** ≤ 5 — 0,6 / 0,2 / 0,2 / 0; 6–19 — 0,5 / 0,2 / 0,3 / 0; 20+ — 0,5 / 0,3 / 0,1 / 0,1.
  ItemKNN — главный во всех корзинах; «больше content новичкам» не подтвердилось (3.3). На всём каталоге веса уходят в
  content (TF-IDF и e5-base 0,6–0,9) — следствие утечки каталога.

## 4. Способ 2 — двухэтапная схема (CatBoost, YetiRank)

Ранжировщик учится на 80% пользователей A, ранняя остановка по NDCG@10 — на остальных 20% A; проверка — на B. В обучении
только пользователи, у которых среди кандидатов есть попадание (остальные не дают сигнала). В режиме «весь каталог» — без
года издания (`i_age`).

In [6]:
DROP = {"известные": [], "весь": ["i_age"]}
cut = int(len(A) * 0.8)
A_fit, A_es = A[:cut], A[cut:]
RANKERS, imp = {}, {}
for regime in REGIMES:
    c = cands_val[regime]
    cols = [x for x in ranker.feature_columns(c) if x not in DROP[regime]]
    RANKERS[regime] = timed(f"обучение ранжировщика ({regime})", "val", lambda: ranker.fit(c, A_fit, A_es, cols=cols, iterations=2000))
    imp[regime] = pd.Series(RANKERS[regime].get_feature_importance(type="PredictionValuesChange"),
                            RANKERS[regime].feature_names_).sort_values(ascending=False)
    print(f"{regime}: деревьев {RANKERS[regime].tree_count_}")
pd.DataFrame(imp).round(1)

Groupwise loss function. OneHotMaxSize set to 10


0:	test: 0.1290290	best: 0.1290290 (0)	total: 528ms	remaining: 17m 35s


200:	test: 0.2101545	best: 0.2119360 (162)	total: 1m 12s	remaining: 10m 47s


400:	test: 0.2217446	best: 0.2221528 (388)	total: 2m 37s	remaining: 10m 29s


Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.2221527611
bestIteration = 388

Shrink model to first 389 iterations.
известные: деревьев 389


Groupwise loss function. OneHotMaxSize set to 10


0:	test: 0.1080552	best: 0.1080552 (0)	total: 636ms	remaining: 21m 11s


200:	test: 0.2052250	best: 0.2075920 (161)	total: 2m 4s	remaining: 18m 37s


400:	test: 0.2156187	best: 0.2177877 (384)	total: 4m 29s	remaining: 17m 56s


Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.2211803711
bestIteration = 423

Shrink model to first 424 iterations.
весь: деревьев 424


,известные,весь
als_pair,6.8000,2.7000
als_rank,2.9000,2.2000
als_score,1.6000,1.3000
author_cnt,4.5000,10.3000
author_last5,1.6000,2.6000
e5_rank,3.1000,4.4000
e5_score,0.7000,1.4000
e5_sim,5.8000,6.1000
i_age,4.8000,NaN
i_days,12.4000,9.0000


In [7]:
def single(lst, users):
    return head(lst, users)

val_rows, val_pu = [], {}
def record(store, rows, part_name, regime, name, reco, tr, part, users):
    table, pu = evaluate(reco, tr, part, users)
    store[(regime, name)] = pu
    rows.append(with_ci(table, pu).assign(часть=part_name, каталог=regime, модель=name))

for regime in REGIMES:
    method, m = BEST_FORMULA[regime]
    raw = cands_val[regime][cands_val[regime][U].isin(set(B))]
    rs = ranker.predict(RANKERS[regime], raw)   # до замены i_wr: ранжировщик учился с m = 20
    cB = with_components(raw, method, m, train, VAL_START)
    L = lists_for(regime, lists_val, content_val)
    record(val_pu, val_rows, "val B", regime, "ItemKNN", single(L["knn"], B), train, val, B)
    record(val_pu, val_rows, "val B", regime, f"TF-IDF, N={N_BEST[(regime, 'tfidf')]}", single(L["tfidf"], B), train, val, B)
    record(val_pu, val_rows, "val B", regime, f"e5-base, N={N_BEST[(regime, 'e5')]}", single(L["e5"], B), train, val, B)
    fs = formula_score(cB, FORMULA[(regime, method, m)])
    for cap in [None, 2]:
        sfx = "" if cap is None else ", ≤ 2 книг автора"
        record(val_pu, val_rows, "val B", regime, f"Формула{sfx}", blend.top_k(cB, fs, max_per_author=cap), train, val, B)
        record(val_pu, val_rows, "val B", regime, f"Двухэтапная{sfx}", blend.top_k(cB, rs, max_per_author=cap), train, val, B)

res_B = pd.concat(val_rows, ignore_index=True)
display(res_B.pivot_table(index=["каталог", "модель"], columns="разрез", values="NDCG@10", sort=False)[["все", "старые", "новые"]])

разрез                                    все  старые  новые
каталог   модель                                            
известные ItemKNN                      0.0101  0.0155 0.0000
          TF-IDF, N=3                  0.0049  0.0075 0.0002
          e5-base, N=1                 0.0051  0.0076 0.0002
          Формула                      0.0116  0.0178 0.0000
          Двухэтапная                  0.0229  0.0357 0.0000
          Формула, ≤ 2 книг автора     0.0100  0.0154 0.0000
          Двухэтапная, ≤ 2 книг автора 0.0215  0.0338 0.0000
весь      ItemKNN                      0.0101  0.0155 0.0000
          TF-IDF, N=3                  0.0147  0.0065 0.0267
          e5-base, N=1                 0.0147  0.0068 0.0259
          Формула                      0.0171  0.0114 0.0241
          Двухэтапная                  0.0373  0.0272 0.0503
          Формула, ≤ 2 книг автора     0.0123  0.0093 0.0157
          Двухэтапная, ≤ 2 книг автора 0.0319  0.0273 0.0365

**Ранжировщик CatBoost YetiRank (обучение на 80% A, ранняя остановка по 20% A, проверка на B):**
- **Без утечки в проверку:** признаки — только по обучению до даты (статистика книги — по оценкам до даты, а не из
  books_clean); метки — попадания валидации; B и test в обучении не участвуют. Финал: веса формулы заново на всём val,
  ранжировщик переобучен на всём val с числом деревьев, найденным на A (389 / 424); test — признаки по train + val,
  дата отсечки 01.01.2022.
- **На B:** двухэтапная 0,0229 против формулы 0,0116 и ItemKNN 0,0101 (+0,0128 [+0,0111; +0,0146]).
- **Главные признаки** — динамика популярности: оценок за последний год (14%), дней с первой оценки (12%); затем
  баллы ALS и content для любой пары (6–7% каждый), возраст книги, давность последней оценки пользователя, автор.
- **Проверка, что это не одна мода:** «популярное за последний год / 90 дней» само по себе — 0,0054–0,0058 на test
  (вдвое хуже ItemKNN). Выигрыш — от сочетания личных сигналов с динамикой популярности, которой нет у ItemKNN.

In [8]:
def sig_table(store, pairs, slices=("все", "старые", "новые"), users_bucket=None):
    rows = []
    for regime, a, b in pairs:
        for s in slices:
            x, y = store[(regime, a)][s], store[(regime, b)][s]
            r = paired_bootstrap(x, y)
            rows.append({"каталог": regime, "сравнение": f"{a} − {b}", "разрез": s, "разница NDCG@10": r["разница"],
                         "95% ДИ": f"{r['low']:+.4f} … {r['high']:+.4f}", "значимо": "да" if r["low"] > 0 or r["high"] < 0 else "нет"})
    return pd.DataFrame(rows)

def tfidf_name(regime): return f"TF-IDF, N={N_BEST[(regime, 'tfidf')]}"
PAIRS = [(r, a, b) for r in REGIMES for a, b in [("Формула", "ItemKNN"), ("Двухэтапная", "ItemKNN"), ("Двухэтапная", "Формула"),
                                                 ("Формула", tfidf_name(r)), ("Двухэтапная", tfidf_name(r)),
                                                 ("Формула, ≤ 2 книг автора", "Формула"),
                                                 ("Двухэтапная, ≤ 2 книг автора", "Двухэтапная")]]
sig_table(val_pu, PAIRS).style.format({"разница NDCG@10": "{:+.4f}"}).hide(axis="index")

каталог,сравнение,разрез,разница NDCG@10,95% ДИ,значимо
известные,Формула − ItemKNN,все,+0.0015,+0.0007 … +0.0024,да
известные,Формула − ItemKNN,старые,+0.0023,+0.0010 … +0.0037,да
известные,Формула − ItemKNN,новые,+0.0000,+0.0000 … +0.0001,нет
известные,Двухэтапная − ItemKNN,все,+0.0128,+0.0111 … +0.0146,да
известные,Двухэтапная − ItemKNN,старые,+0.0202,+0.0175 … +0.0228,да
известные,Двухэтапная − ItemKNN,новые,+0.0000,+0.0000 … +0.0000,нет
известные,Двухэтапная − Формула,все,+0.0113,+0.0096 … +0.0130,да
известные,Двухэтапная − Формула,старые,+0.0178,+0.0153 … +0.0203,да
известные,Двухэтапная − Формула,новые,-0.0000,-0.0001 … +0.0000,нет
известные,"Формула − TF-IDF, N=3",все,+0.0067,+0.0055 … +0.0079,да


**Правило «не больше 2 книг одного автора» (для сайта остаётся):** на test в «известных» стоит двухэтапной
−0,0011 (−4%, 0,0248 → 0,0237), формуле −0,0016 (−14%, 0,0120 → 0,0103 — ниже ItemKNN без правила); оба значимо.
Двухэтапная с правилом всё равно в 2,2 раза лучше ItemKNN. На всём каталоге правило дороже (−13% и −28%).

## 5. Тест — один прогон

Обучение — до 2022 (обучение + валидация), каталог content — известные до 2022 (или весь), N профиля, нормировка и m — с A;
веса формулы — подобраны заново на всей валидации; ранжировщик — обучен на всей валидации (A + B) с числом деревьев из A.

In [9]:
# Финальные веса формулы и ранжировщики — на всей валидации
FINAL_W, FINAL_R = {}, {}
for regime in REGIMES:
    method, m = BEST_FORMULA[regime]
    FINAL_W[regime] = tune_formula(with_components(cands_val[regime], method, m, train, VAL_START), val, users_val)
    cols = list(RANKERS[regime].feature_names_)
    FINAL_R[regime] = timed(f"обучение ранжировщика ({regime})", "test",
                            lambda: ranker.fit(cands_val[regime], users_val, cols=cols, iterations=RANKERS[regime].tree_count_))
display(pd.DataFrame({r: {b: " / ".join(f"{x:.1f}" for x in FINAL_W[r][b][0]) for b in FINAL_W[r]} for r in REGIMES})
        .rename_axis("корзина (веса knn / tfidf / e5 / wr)"))
del cands_val, content_val
gc.collect()

Groupwise loss function. OneHotMaxSize set to 10


0:	total: 1.5s	remaining: 9m 40s


200:	total: 4m 6s	remaining: 3m 50s


388:	total: 7m 21s	remaining: 0us


Groupwise loss function. OneHotMaxSize set to 10


0:	total: 1.1s	remaining: 7m 45s


200:	total: 4m 36s	remaining: 5m 7s


400:	total: 9m	remaining: 31s


423:	total: 9m 28s	remaining: 0us


,известные,весь
корзина (веса knn / tfidf / e5 / wr),,
≤ 5,0.6 / 0.2 / 0.2 / 0.0,0.3 / 0.3 / 0.3 / 0.1
6–19,0.5 / 0.2 / 0.3 / 0.0,0.0 / 0.4 / 0.5 / 0.1
20+,0.5 / 0.3 / 0.1 / 0.1,0.2 / 0.5 / 0.2 / 0.1


7541

In [10]:
lists_test, als_test = cf_lists(train_full, users_test, "test")
test_rows, test_pu, cands_test = [], {}, {}
for regime in REGIMES:
    mask = catalog_mask(regime, TEST_START)
    L = dict(lists_test)
    for s in ["tfidf", "e5"]:
        n = N_BEST[(regime, s)]
        L[s] = timed(f"кандидаты {s} ({regime})", "test", lambda: recommend(VECTORS[s], ids_all, train_full, users_test, k=K_LIST, allowed=mask,
                                                              profile_from=truncate_history(train_full, users_test, n)))
        L[s] = candidates.drop_survivors(L[s], train_full)
    c = timed(f"признаки ({regime})", "test", lambda: build_cands(regime, L, als_test, train_full, test, users_test, TEST_START))
    rs = timed(f"переранжирование ({regime})", "test", lambda: ranker.predict(FINAL_R[regime], c))
    method, m = BEST_FORMULA[regime]
    c = with_components(c, method, m, train_full, TEST_START)
    fs = formula_score(c, FINAL_W[regime])
    record(test_pu, test_rows, "test", regime, "ItemKNN", single(L["knn"], users_test), train_full, test, users_test)
    record(test_pu, test_rows, "test", regime, tfidf_name(regime), single(L["tfidf"], users_test), train_full, test, users_test)
    record(test_pu, test_rows, "test", regime, f"e5-base, N={N_BEST[(regime, 'e5')]}", single(L["e5"], users_test), train_full, test, users_test)
    for cap in [None, 2]:
        sfx = "" if cap is None else ", ≤ 2 книг автора"
        record(test_pu, test_rows, "test", regime, f"Формула{sfx}", blend.top_k(c, fs, max_per_author=cap), train_full, test, users_test)
        record(test_pu, test_rows, "test", regime, f"Двухэтапная{sfx}", blend.top_k(c, rs, max_per_author=cap), train_full, test, users_test)
    cands_test[regime] = c[[U, "u_n"]].drop_duplicates(U)
    print(f"{regime}: {len(c):,} пар, попаданий в кандидатах {candidates.recall_of_candidates(c, test, users_test):.1%}")
    del c; gc.collect()

res_test = pd.concat(test_rows, ignore_index=True)
cols_out = ["часть", "каталог", "модель", "разрез", "пользователей", "NDCG@10", "NDCG@10 95% ДИ", "Recall@10", "Coverage@10", "Novelty@10"]
pd.concat([res_B, res_test])[cols_out].to_csv(REPORTS / "hybrid.csv", index=False, encoding="utf-8")
display(res_test.pivot_table(index=["каталог", "модель"], columns="разрез", values="NDCG@10", sort=False)[["все", "старые", "новые"]])

известные: 6,867,040 пар, попаданий в кандидатах 9.6%


весь: 6,874,945 пар, попаданий в кандидатах 13.8%


разрез                                    все  старые  новые
каталог   модель                                            
известные ItemKNN                      0.0106  0.0160 0.0000
          TF-IDF, N=3                  0.0046  0.0066 0.0004
          e5-base, N=1                 0.0048  0.0069 0.0003
          Формула                      0.0120  0.0179 0.0000
          Двухэтапная                  0.0248  0.0373 0.0000
          Формула, ≤ 2 книг автора     0.0103  0.0156 0.0000
          Двухэтапная, ≤ 2 книг автора 0.0237  0.0357 0.0000
весь      ItemKNN                      0.0106  0.0160 0.0000
          TF-IDF, N=3                  0.0140  0.0060 0.0253
          e5-base, N=1                 0.0129  0.0064 0.0218
          Формула                      0.0162  0.0104 0.0236
          Двухэтапная                  0.0377  0.0266 0.0512
          Формула, ≤ 2 книг автора     0.0116  0.0082 0.0154
          Двухэтапная, ≤ 2 книг автора 0.0328  0.0273 0.0377

In [11]:
sig_test = sig_table(test_pu, PAIRS)
sig_test.to_csv(REPORTS / "hybrid_test_significance.csv", index=False, encoding="utf-8")
sig_test.style.format({"разница NDCG@10": "{:+.4f}"}).hide(axis="index")

каталог,сравнение,разрез,разница NDCG@10,95% ДИ,значимо
известные,Формула − ItemKNN,все,+0.0013,+0.0007 … +0.0020,да
известные,Формула − ItemKNN,старые,+0.0019,+0.0010 … +0.0028,да
известные,Формула − ItemKNN,новые,+0.0000,+0.0000 … +0.0001,нет
известные,Двухэтапная − ItemKNN,все,+0.0141,+0.0126 … +0.0155,да
известные,Двухэтапная − ItemKNN,старые,+0.0213,+0.0193 … +0.0232,да
известные,Двухэтапная − ItemKNN,новые,+0.0000,+0.0000 … +0.0000,нет
известные,Двухэтапная − Формула,все,+0.0128,+0.0115 … +0.0142,да
известные,Двухэтапная − Формула,старые,+0.0194,+0.0175 … +0.0215,да
известные,Двухэтапная − Формула,новые,-0.0000,-0.0001 … +0.0000,нет
известные,"Формула − TF-IDF, N=3",все,+0.0073,+0.0064 … +0.0082,да


In [12]:
# По корзинам числа оценок пользователя (обучение + валидация), разрез «все»
u_n = cands_test["известные"].set_index(U)["u_n"]
bucket = blend.bucket_of(u_n)
rows = []
for regime in REGIMES:
    for (rg, name), pu in test_pu.items():
        if rg != regime or "автора" in name:
            continue
        for _, _, b in blend.BUCKETS:
            x = pu["все"][pu["все"].index.isin(bucket[bucket == b].index)]
            lo, hi = bootstrap_ci(x)
            rows.append({"каталог": regime, "модель": name, "корзина": b, "пользователей": len(x), "NDCG@10": x.mean(),
                         "95% ДИ": f"{lo:.4f}–{hi:.4f}"})
by_bucket = pd.DataFrame(rows)
by_bucket.to_csv(REPORTS / "hybrid_test_buckets.csv", index=False, encoding="utf-8")
display(by_bucket.pivot_table(index=["каталог", "модель"], columns="корзина", values="NDCG@10", sort=False)[[b for *_, b in blend.BUCKETS]])

brow = []
for regime in REGIMES:
    for a, b_ in [("Двухэтапная", "ItemKNN"), ("Формула", "ItemKNN"), ("Двухэтапная", "Формула")]:
        for _, _, b in blend.BUCKETS:
            idx = bucket[bucket == b].index
            x, y = test_pu[(regime, a)]["все"], test_pu[(regime, b_)]["все"]
            r = paired_bootstrap(x[x.index.isin(idx)], y[y.index.isin(idx)])
            brow.append({"каталог": regime, "сравнение": f"{a} − {b_}", "корзина": b, "разница NDCG@10": r["разница"],
                         "95% ДИ": f"{r['low']:+.4f} … {r['high']:+.4f}", "значимо": "да" if r["low"] > 0 or r["high"] < 0 else "нет"})
pd.DataFrame(brow).style.format({"разница NDCG@10": "{:+.4f}"}).hide(axis="index")

корзина                   ≤ 5   6–19    20+
каталог   модель                           
известные ItemKNN      0.0099 0.0107 0.0131
          TF-IDF, N=3  0.0043 0.0044 0.0068
          e5-base, N=1 0.0048 0.0046 0.0057
          Формула      0.0107 0.0121 0.0162
          Двухэтапная  0.0207 0.0270 0.0337
весь      ItemKNN      0.0099 0.0107 0.0131
          TF-IDF, N=3  0.0107 0.0157 0.0215
          e5-base, N=1 0.0117 0.0135 0.0160
          Формула      0.0143 0.0164 0.0231
          Двухэтапная  0.0299 0.0434 0.0510

каталог,сравнение,корзина,разница NDCG@10,95% ДИ,значимо
известные,Двухэтапная − ItemKNN,≤ 5,+0.0108,+0.0089 … +0.0125,да
известные,Двухэтапная − ItemKNN,6–19,+0.0163,+0.0139 … +0.0189,да
известные,Двухэтапная − ItemKNN,20+,+0.0206,+0.0166 … +0.0244,да
известные,Формула − ItemKNN,≤ 5,+0.0008,+0.0001 … +0.0015,да
известные,Формула − ItemKNN,6–19,+0.0013,+0.0003 … +0.0025,да
известные,Формула − ItemKNN,20+,+0.0031,+0.0013 … +0.0051,да
известные,Двухэтапная − Формула,≤ 5,+0.0100,+0.0083 … +0.0117,да
известные,Двухэтапная − Формула,6–19,+0.0149,+0.0126 … +0.0173,да
известные,Двухэтапная − Формула,20+,+0.0175,+0.0138 … +0.0213,да
весь,Двухэтапная − ItemKNN,≤ 5,+0.0199,+0.0174 … +0.0224,да


In [13]:
# Время: обучение (с) и выдача на пользователя (мс)
t = pd.DataFrame(timing)
display(t.groupby(["часть", "этап"], sort=False)["с"].sum().to_frame())
infer = t[(t["часть"] == "test") & t["этап"].str.match("кандидаты|признаки|переранжирование")]
per_regime = {r: infer[~infer["этап"].str.contains("весь" if r == "известные" else "известные")] for r in REGIMES}
for r, x in per_regime.items():
    # кандидаты ItemKNN и ALS общие для обоих режимов
    print(f"{r}: выдача (кандидаты + признаки + переранжирование) — {x['с'].sum():.0f} с на {len(users_test):,} пользователей, "
          f"{1000 * x['с'].sum() / len(users_test):.1f} мс на пользователя (пачкой)")

с
часть этап                                       
val   обучение ItemKNN                     2.9000
      обучение ALS                        11.9000
      кандидаты ItemKNN                    3.2000
      кандидаты ALS                        4.7000
      кандидаты tfidf                   2021.5000
      кандидаты e5                       237.2000
      признаки (известные)               111.6000
      признаки (весь)                    123.9000
      обучение ранжировщика (известные)  197.5000
      обучение ранжировщика (весь)       363.5000
test  обучение ранжировщика (известные)  447.1000
      обучение ранжировщика (весь)       573.2000
      обучение ItemKNN                     3.6000
      обучение ALS                        14.2000
      кандидаты ItemKNN                    3.3000
      кандидаты ALS                        5.7000
      кандидаты tfidf (известные)        317.5000
      кандидаты e5 (известные)            30.5000
      признаки (известные)               112.1000
      переранжирование (известные)         3.5000
      кандидаты tfidf (весь)             320.0000
      кандидаты e5 (весь)                 29.7000
      признаки (весь)                     94.5000
      переранжирование (весь)              3.1000

известные: выдача (кандидаты + признаки + переранжирование) — 473 с на 20,009 пользователей, 23.6 мс на пользователя (пачкой)
весь: выдача (кандидаты + признаки + переранжирование) — 456 с на 20,009 пользователей, 22.8 мс на пользователя (пачкой)


**Тест (один прогон, 20 009 пользователей), режим «известные на дату» — основной:**
- **ItemKNN 0,0106, TF-IDF (N = 3) 0,0046, e5-base (N = 1) 0,0048, формула 0,0120, двухэтапная 0,0248.**
  Формула − ItemKNN +0,0013 [+0,0007; +0,0020], двухэтапная − ItemKNN **+0,0141 [+0,0126; +0,0155]**, двухэтапная −
  формула +0,0128 — всё значимо. Валидация B → test: 0,0229 → 0,0248, формула 0,0116 → 0,0120 — переобучения нет.
- **По корзинам** (≤ 5 / 6–19 / 20+): двухэтапная 0,0207 / 0,0270 / 0,0337 против ItemKNN 0,0099 / 0,0107 / 0,0131 —
  значимо лучше в каждой, выигрыш растёт с историей.
- **«Новые» книги — 0 у всех моделей:** честно рекомендовать книгу без оценок офлайн на этих данных нельзя (3.2).
- **Весь каталог (верхняя граница, ранжировщик без года и возраста):** двухэтапная 0,0377 (на «новых» 0,0512) —
  завышено утечкой каталога, в сравнение не берём.
- **Время:** обучение ItemKNN 4 с, ALS 14 с, ранжировщика 7,5 мин (389 деревьев, 7,3 млн пар, процессор).
  Выдача пачкой — **24 мс на пользователя**: кандидаты TF-IDF 16 мс (самое дорогое), признаки 5,6 мс,
  e5-base 1,5 мс, переранжирование 0,2 мс.